# ClassicML — Classical ML for Fato/Tese/Ruído

Runs 8 classical models (LR, SVM, Random Forest, Extra Trees, XGBoost, LightGBM, Naive Bayes)
with stratified K-Fold (5 folds), all metrics from the spreadsheet, and PT-BR/EN figures.

**GPU is optional** (XGBoost can use it if available). Everything runs on CPU.

In [ ]:
import os, sys

PROJECT_ROOT = '/content/drive/MyDrive/Fato_Tese_Ruido'
CLASSIC_ROOT = os.path.join(PROJECT_ROOT, 'ClassicML')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not running in Colab — using local paths.')

for p in [os.path.join(PROJECT_ROOT, 'src'), CLASSIC_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ['FTR_PROJECT_ROOT'] = PROJECT_ROOT
os.chdir(CLASSIC_ROOT)
print('CLASSIC_ROOT =', CLASSIC_ROOT)

In [ ]:
if IN_COLAB:
    !pip install -q xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn openpyxl pyyaml
    print('Dependencies installed.')

In [ ]:
# Validate code and data before spending CPU time
import os

checks = {
    'ftr_classic/__init__.py': 'ClassicML package',
    'configs/_base.yaml':      'base config',
    '../data/processed':       'BERT splits (reused)',
    '../Data/dataset_FINAL.json': 'raw dataset',
}
missing = []
for rel, desc in checks.items():
    ok = os.path.exists(rel) or os.path.exists(os.path.join(CLASSIC_ROOT, rel))
    print(f'  [{"OK" if ok else "MISSING"}] {rel}  ({desc})')
    if not ok:
        missing.append(rel)
if missing:
    raise SystemExit('Fix the items above before continuing.')

In [ ]:
# Parameters
SMOKE_TEST   = False  # @param {type:"boolean"} — True = 2 folds, 500 samples
ONLY_MODELS  = []     # @param — empty list = all. E.g.: ['logistic_regression','xgboost']
NON_INTERACTIVE = True  # @param {type:"boolean"}

In [ ]:
import glob, traceback
from ftr_classic.trainer import run_experiment
from ftr_classic.config import load_config

configs = sorted(glob.glob('configs/*.yaml'))
configs = [c for c in configs if not c.endswith('_base.yaml')]
if ONLY_MODELS:
    configs = [c for c in configs if any(m in c for m in ONLY_MODELS)]

if SMOKE_TEST:
    print('SMOKE_TEST active — 2 folds, max 500 samples per experiment')

results = []
for cfg_path in configs:
    eid = cfg_path.split('/')[-1].replace('.yaml','')
    print(f'\n{"="*60}\n>>> {eid}\n{"="*60}')
    try:
        cfg = load_config(cfg_path)
        if SMOKE_TEST:
            cfg['cv']['n_splits'] = 2
        status = run_experiment(cfg_path, PROJECT_ROOT, CLASSIC_ROOT + '/experiments')
        results.append((eid, status['status'], status.get('f1_macro_mean')))
        if status.get('f1_macro_mean'):
            print(f'  f1_macro: {status["f1_macro_mean"]:.4f} ± {status.get("f1_macro_std",0):.4f}')
    except Exception as e:
        print(f'ERROR in {eid}: {e}')
        traceback.print_exc()
        results.append((eid, 'failed', None))

print('\n=== SUMMARY ===')
for eid, st, f1 in results:
    f1_str = f'{f1:.4f}' if f1 else 'N/A'
    print(f'  {eid:25s} -> {st:10s}  f1_macro={f1_str}')

In [ ]:
# Consolidate results and generate comparative table + figures
import subprocess
result = subprocess.run(['python', 'scripts/consolidate.py'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-300:])

import pandas as pd
df = pd.read_csv('results/comparison_table.csv')
cols = ['experiment_id','f1_macro_mean','f1_macro_std','test_f1_macro','test_accuracy','test_mcc']
df[[c for c in cols if c in df.columns]].sort_values('f1_macro_mean', ascending=False)

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*_pt.png')):
    print(f)
    display(Image(f))